# Federated Learning (Student)

This notebook is a demo using Flower simulation. You will run the federated learning simulation with multiple clients, then compare the results.

Please Fill in the information below:

------
NRP  : XXXXXXXXXX \
Name : XXXXXXXXXX

------

## 1) Setup the Logger

Sign up into Weights & Biases (W&B) at [W&B Sign Up](https://app.wandb.ai/login?signup=true), create a new project, and get your API key from the W&B dashboard. Then, set up the W&B logger in your simulation configuration.

In [ ]:
# Install Weights & Biases (W&B) for logging
!pip install wandb weave

Setup the API key in Google Colab by adding new Secret in the left sidebar, then Add New Secret with the key `WANDB_API_KEY` and the value of your API key. Then, you can access the API key in your code using `os.environ["WANDB_API_KEY"]`.

In [ ]:
# Load the saved API key from Google Colab secrets
from google.colab import userdata
wandb_key = userdata.get('WANDB_API_KEY')

# Set the API key as an environment variable
import os
os.environ['WANDB_API_KEY'] = wandb_key

## 2) Create the project

Use the quickstart to create the project. Main source from the [Flower documentation - How to Run Simulations](https://flower.ai/docs/framework/how-to-run-simulations.html).

In [ ]:
# Install flwr
!pip install flwr flwr-datasets[vision]

If you prefer to use the quickstart setup using quickstart pytorch template, you can run the following command to create a new project with the necessary files and structure for a federated learning simulation using PyTorch. This will generate a ready-made Flower project that you can customize and run.

```shell
flwr new @flwrlabs/quickstart-pytorch
```

But for this demo, we will use the advanced-pytorch template, which available on GitHub at [Flower Advanced PyTorch Template](https://github.com/flwrlabs/flower/tree/main/examples/advanced-pytorch). We can clone the repository and copy the necessary files to our project directory.

In [ ]:
# Download the Flower repository's main branch as a zip file
!wget https://github.com/muhammadariffaizin/flwr-experiments/archive/refs/heads/main.zip -O flower-main.zip

# Unzip the downloaded file into a temporary directory
!unzip -o flower-main.zip -d flower-repo

# Move the 'advanced-pytorch' example directory to the current working directory
!mv flower-repo/flwr-experiments-main/advanced-pytorch advanced-pytorch

# Clean up the temporary repository and zip file
!rm -rf flower-repo flower-main.zip

This will create a new directory called `advanced-pytorch` with the following structure:

```shell
advanced-pytorch
├── pytorch_example
│   ├── __init__.py
│   ├── client_app.py   # Defines your ClientApp
│   ├── server_app.py   # Defines your ServerApp
│   ├── strategy.py     # Defines a custom strategy
│   └── task.py         # Defines your model, training and data loading
├── pyproject.toml      # Project metadata like dependencies and configs
└── README.md
```

In [ ]:
# Change directory to the project
%cd advanced-pytorch

Install the package and run the simulation. You should see the training progress in the output.
There is two `flwr` packages: `flwr` and `flwr[simulation]`. The former is the core package for federated learning, while the latter includes additional dependencies for running simulations. In this quickstart, we will use the `flwr[simulation]` package to run the federated learning simulation.
This quickstart basically install the `flwr[simulation]`, `flwr-datasets[vision]`, `torch`, and `torchvision` packages.

In [ ]:
# Install the package
!pip install -e .

## 3) Project Configuration

The `pyproject.toml` file contains the configuration for the federated learning simulation.

In [ ]:
# Read the current content of the pyproject.toml file
!cat pyproject.toml

In [ ]:
PARTITIONER = "dirichlet"
DATASET_NAME = "uoft-cs/cifar100"
NUM_CLIENTS = 20
FEATURE_COLUMN = "img"
TARGET_COLUMN = "fine_label"

You can modify this file to change the parameters of the simulation, such as the number of rounds, the fraction of clients used for training and evaluation, and the local epochs.

In [ ]:
from IPython.core.magic import register_line_cell_magic

@register_line_cell_magic
def writetemplate(line, cell):
    with open(line, 'w') as f:
        # This replaces {var_name} with the actual variable value
        f.write(cell.format(**globals()))

In [ ]:
%%writetemplate pyproject.toml
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "pytorch-example"
version = "1.0.0"
description = "Federated Learning with PyTorch and Flower (Advanced Example)"
license = "Apache-2.0"
dependencies = [
    "flwr[simulation]>=1.28.0",
    "flwr-datasets[vision]>=0.6.0",
    "torch>=2.8.0",
    "torchvision>=0.23.0",
    "wandb==0.17.8",
]

[tool.hatch.build.targets.wheel]
packages = ["."]

[tool.flwr.app]
publisher = "muhammadariffaizin"

[tool.flwr.app.components]
serverapp = "pytorch_example.server_app:app"
clientapp = "pytorch_example.client_app:app"

[tool.flwr.app.config]
num-server-rounds = 10
fraction-train = 0.25
fraction-evaluate = 0.5
local-epochs = 1
server-device = "cpu"
initial-lr = 0.1
model-fn = "pytorch_example.model:create_model"
personalized-layer-name = "fc2"
load-data-fn = "pytorch_example.data_custom:load_custom_data"
dataset-name = "{DATASET_NAME}"
batch-size = 32
feature-column = "{FEATURE_COLUMN}"
target-column = "{TARGET_COLUMN}"
partitioner-id = "{PARTITIONER}"
partition-alpha = 1.0
partition-seed = 42
num-shards-per-partition = 2

You can also specify the partitioner to use for splitting the data among clients. The available partitioners are `iid`, `dirichlet`, and `shard`. Each partitioner creates a different distribution of data across clients, which can affect the performance of the federated learning algorithm.

In [ ]:
!flwr federation simulation-config --num-supernodes={NUM_CLIENTS}

The next important task is configuring the model architecture, in this demo the model architecture is defined in the `model.py` file.

In [ ]:
# Show the current content of the model.py file
!cat pytorch_example/model.py

In [ ]:
# Show the predefined model architecture
!cat pytorch_example/models/{DATASET_NAME.split('/')[-1]}.py

Make an update to the `model.py` file to define your model architecture. You can use any PyTorch model architecture that is suitable for your dataset and task.

In [ ]:
!cp pytorch_example/models/{DATASET_NAME.split('/')[-1]}.py pytorch_example/model.py

## 4) Data Distribution Visualization using a partitioner

In simulation, you can choose how to split the data among clients using a partitioner. The partitioner determines how the data is distributed across clients, which can affect the performance of the federated learning algorithm.

1) IID partitioner: This partitioner splits the data randomly and uniformly across clients, resulting in each client having a similar distribution of data.
2) Dirichlet partitioner: This partitioner uses a Dirichlet distribution to create non-IID data splits, where each client has a different distribution of data.
3) Shard partitioner: This partitioner shuffles the data and then splits it into shards (i.e., subsets), which are then assigned to clients.

Change `PARTITIONER` to try a different split: `iid`, `dirichlet`, or `shard`.

In [ ]:
from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import IidPartitioner, DirichletPartitioner, ShardPartitioner

def make_partitioner(kind: str, num_partitions: int = 10):
    if kind == "iid":
        return IidPartitioner(num_partitions=num_partitions)
    if kind == "dirichlet":
        return DirichletPartitioner(
            num_partitions=num_partitions,
            partition_by="fine_label",
            alpha=0.3,
            seed=42,
            min_partition_size=0,
        )
    if kind == "shard":
        return ShardPartitioner(
            num_partitions=num_partitions,
            partition_by="fine_label",
            num_shards_per_partition=2,
        )
    raise ValueError(f"Unknown partitioner: {kind}")

partitioner = make_partitioner(PARTITIONER, NUM_CLIENTS)
fds = FederatedDataset(
    dataset=DATASET_NAME,
    partitioners={"train": partitioner},
)

The flwr package provides a utility function to visualize the data distribution across clients. You can use the `plot_label_distributions` function to see how the data is distributed among clients for each partitioner.

Re-run this after changing `PARTITIONER`.

In [ ]:
from flwr_datasets.visualization import plot_label_distributions

fig, ax, df = plot_label_distributions(
    fds.partitioners["train"],
    label_name="fine_label",
    plot_type="bar",
    size_unit="absolute",
    partition_id_axis="x",
    legend=True,
    verbose_labels=True,
    title=f"Per-Client Label Distribution ({PARTITIONER})"
 )

## 5) Run federated learning

To run the federated learning simulation, use the `flwr run` command with the appropriate configuration. You can specify various parameters to control the training process.

There are several parameters you can change to see how they affect the training process. The `run-config` flag allows you to specify a comma-separated list of key-value pairs that configure the training process. Here are some of the parameters you can change:
- `num-server-rounds`: The number of rounds of federated learning to run. Each round consists of a server selecting a subset of clients, sending them the current global model, and then aggregating their updates to form a new global model.
- `fraction-train`: The fraction of clients to use for training in each round. For example, if you have 100 clients and set `fraction-train=0.5`, then 50 clients will be selected for training in each round.
- `fraction-evaluate`: The fraction of clients to use for evaluation in each round. For example, if you have 100 clients and set `fraction-evaluate=0.2`, then 20 clients will be selected for evaluation in each round.

In [ ]:
!flwr run . --stream --run-config "num-server-rounds=5 fraction-train=0.5"

## 6) Task

Write 4-6 short sentences about what you observed. Mention:
- Which run was centralized-like vs federated
- Differences in accuracy/loss trend and runtime
- One thing you found surprising